In [ ]:
import csv
import glob
import itertools
import json
import math
import os
import random
import shutil
import sys
import time
import warnings
import nibabel as nib
import numpy as np
import pandas as pd
import pingouin as pg
import seaborn as sns
import statsmodels.api as sm
from PIL import Image
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
from prettytable import PrettyTable
from scipy import ndimage
from scipy.stats import ttest_ind, ttest_rel
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.tools.sm_exceptions import ConvergenceWarning

In [ ]:
"""
Defines a lookup dictionary mapping cortical parcel label IDs to their corresponding FreeSurfer/DKT-style region names.

This table is used to translate numeric cortical segmentation labels into readable anatomical region names for downstream summaries, analyses, and visualizations.
"""

label_dict={
  1:  'bankssts',                        
  2:  'caudalanteriorcingulate',         
  3:  'caudalmiddlefrontal',             
  4:  'corpuscallosum',                 
  5:  'cuneus',                         
  6:  'entorhinal',                     
  7:  'fusiform',                        
  8:  'inferiorparietal',               
  9:  'inferiortemporal',               
 10:  'isthmuscingulate',                
 11:  'lateraloccipital',                
 12:  'lateralorbitofrontal',            
 13:  'lingual',                        
 14:  'medialorbitofrontal',            
 15:  'middletemporal',                 
 16:  'parahippocampal',                  
 17:  'paracentral',                     
 18:  'parsopercularis',                
 19:  'parsorbitalis',                   
 20:  'parstriangularis',              
 21:  'pericalcarine',                  
 22:  'postcentral',                    
 23:  'posteriorcingulate',            
 24:  'precentral',                      
 25:  'precuneus',                      
 26:  'rostralanteriorcingulate',        
 27:  'rostralmiddlefrontal',            
 28:  'superiorfrontal',                 
 29:  'superiorparietal',                 
 30:  'superiortemporal',                
 31:  'supramarginal',                    
 32:  'frontalpole',                    
 33:  'temporalpole',                    
 34:  'transversetemporal',           
 35:  'insula'}

In [ ]:
df = pd.read_csv("")

In [ ]:
"""
Computes a composite deep gray matter (DGM) volume from selected postmortem right-hemisphere limbic and subcortical structures.

The script calculates a volume-weighted DGM summary measure across hippocampus, amygdala, accumbens, caudate, putamen, thalamus, and pallidum, then normalizes the composite volume by antemortem ICV.
"""

# 7 nuclei included in weighted average (select the ones which you are interested in)
dgm_cols = [
    "postmortem_hippocampus",
    "postmortem_amygdala",
    "postmortem_accumbens_area",
    "postmortem_caudate",
    "postmortem_putamen",
    "postmortem_thalamus",
    "postmortem_pallidum"
]

def weighted_dgm(row):
    V = row[dgm_cols].values.astype(float)
    if np.sum(V) == 0:
        return np.nan
    weights = V / np.sum(V)
    return np.sum(weights * V)

# Weighted composite DGM volume
df_use["DGM_weighted"] = df_use.apply(weighted_dgm, axis=1)

# Normalize by ANTEmortem ICV (as requested)
icv_col = "antemortem_icv"

if icv_col in df_use.columns:
    df_use["DGM_weighted_ICVnorm"] = df_use["DGM_weighted"] / df_use[icv_col]
else:
    print("[WARNING] Missing column: antemortem_icv")
 nuclei included in weighted average (select the ones which you are interested in)
dgm_cols = [
    "postmortem_hippocampus",
    "postmortem_amygdala",
    "postmortem_accumbens_area",
    "postmortem_caudate",
    "postmortem_putamen",
    "postmortem_thalamus",
    "postmortem_pallidum"
]

def weighted_dgm(row):
    V = row[dgm_cols].values.astype(float)
    if np.sum(V) == 0:
        return np.nan
    weights = V / np.sum(V)
    return np.sum(weights * V)

# Weighted composite DGM volume
df_use["DGM_weighted"] = df_use.apply(weighted_dgm, axis=1)

# Normalize by ANTEmortem ICV (as requested)
icv_col = "antemortem_icv"

if icv_col in df_use.columns:
    df_use["DGM_weighted_ICVnorm"] = df_use["DGM_weighted"] / df_use[icv_col]
else:
    print("[WARNING] Missing column: antemortem_icv")

# 7 nuclei included in weighted average (select the ones which you are interested in)
dgm_cols = [
    "postmortem_hippocampus",
    "postmortem_amygdala",
    "postmortem_accumbens_area",
    "postmortem_caudate",
    "postmortem_putamen",
    "postmortem_thalamus",
    "postmortem_pallidum"
]

def weighted_dgm(row):
    V = row[dgm_cols].values.astype(float)
    if np.sum(V) == 0:
        return np.nan
    weights = V / np.sum(V)
    return np.sum(weights * V)

# Weighted composite DGM volume
df_use["DGM_weighted"] = df_use.apply(weighted_dgm, axis=1)

# Normalize by ANTEmortem ICV (as requested)
icv_col = "antemortem_icv"

if icv_col in df_use.columns:
    df_use["DGM_weighted_ICVnorm"] = df_use["DGM_weighted"] / df_use[icv_col]
else:
    print("[WARNING] Missing column: antemortem_icv")


In [ ]:
"""
Computes disease-specific partial Spearman correlations between cortical thickness regional measures and an ICV-normalized weighted deep gray matter volume composite.

The script adjusts for age at death, sex, education, and PMI, applies FDR correction across cortical regions within each disease group, and visualizes the resulting correlation coefficients in a single heatmap with FDR significance stars and raw p-values.
"""

sns.set(style="whitegrid", context="talk")

################################################################################
# 1. Cortical regions
################################################################################

cortical_regions = [
    'bankssts','caudalanteriorcingulate','caudalmiddlefrontal',
    'corpuscallosum','cuneus','entorhinal','fusiform','inferiorparietal',
    'inferiortemporal','isthmuscingulate','lateraloccipital',
    'lateralorbitofrontal','lingual','medialorbitofrontal',
    'middletemporal','parahippocampal','paracentral','parsopercularis',
    'parsorbitalis','parstriangularis','pericalcarine','postcentral',
    'posteriorcingulate','precentral','precuneus','rostralanteriorcingulate',
    'rostralmiddlefrontal','superiorfrontal','superiorparietal',
    'superiortemporal','supramarginal','frontalpole','temporalpole',
    'transversetemporal','insula'
]

# Keep only existing columns
cortical_regions = [c for c in cortical_regions if c in df_use.columns]

################################################################################
# 2. Setup
################################################################################

covars = ["AgeatDeath", "Sex", "Education", "PMI"]
group_col = "NPDx1"

disease_groups = [
    "alzheimer's disease",
    "lewy body disease",
    "ftld-tdp",
    "tauopathies"
]

################################################################################
# 3. Partial Spearman per disease × cortical region
################################################################################

rows = []

for disease in disease_groups:
    df_g = df_use[df_use[group_col] == disease]

    for cort in cortical_regions:
        cols_needed = ["DGM_weighted_ICVnorm", cort] + covars
        d = df_g[cols_needed].dropna()
        if len(d) < 10:
            continue

        res = pg.partial_corr(
            data=d,
            x=cort, y="DGM_weighted_ICVnorm",
            covar=covars,
            method="spearman"
        )

        rows.append({
            "Disease": disease,
            "Cortical": cort,
            "rho": res["r"].iloc[0],
            "p_raw": res["p-val"].iloc[0],
            "N": len(d)
        })

results_df = pd.DataFrame(rows)

################################################################################
# 4. Significance levels (raw + FDR)
################################################################################

def sig_stars(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    return ""

results_df["Sig_raw"] = results_df["p_raw"].apply(sig_stars)

# FDR per disease
results_df["Sig_FDR"] = ""
results_df["p_FDR"] = np.nan

for disease in disease_groups:
    mask = results_df["Disease"] == disease
    if mask.sum() == 0:
        continue

    reject, p_fdr = pg.multicomp(results_df.loc[mask, "p_raw"], method="fdr_bh")

    results_df.loc[mask, "p_FDR"] = p_fdr
    results_df.loc[mask, "Sig_FDR"] = [sig_stars(p) for p in p_fdr]

################################################################################
# 5. Build ONE heatmap matrix
################################################################################

# Matrix of ρ values (for color)
rho_mat = np.zeros((len(cortical_regions), len(disease_groups)))
rho_mat[:] = np.nan

# Matrix of annotation strings
annot_mat = np.empty((len(cortical_regions), len(disease_groups)), dtype=object)

for i, cort in enumerate(cortical_regions):
    for j, disease in enumerate(disease_groups):

        row = results_df[(results_df["Disease"] == disease) &
                         (results_df["Cortical"] == cort)]

        if len(row) == 0:
            annot_mat[i, j] = ""
            continue

        rho = row["rho"].iloc[0]
        p_raw = row["p_raw"].iloc[0]
        sig_fdr = row["Sig_FDR"].iloc[0]

        rho_mat[i, j] = rho
        annot_mat[i, j] = f"{rho:.2f}{sig_fdr}\n({p_raw:.1e})"

################################################################################
# 6. Plot SINGLE heatmap (Y=cortical, X=disease)
################################################################################

plt.figure(figsize=(14, 20))
ax = sns.heatmap(
    rho_mat,
    annot=annot_mat,
    fmt="",
    cmap="PRGn",
    vmin=-1, vmax=1,
    cbar_kws={"label": "Partial Spearman ρ"},
    annot_kws={"fontsize": 10}
)

ax.set_xticklabels(disease_groups, rotation=45, ha="right")
ax.set_yticklabels(cortical_regions, rotation=0)
plt.title(
    "Cortical Thickness ↔ DGM_weighted_ICVnorm\n"
    "ρ with FDR stars (superscript) and raw p in brackets",
    fontsize=18
)

plt.tight_layout()
plt.show()


In [ ]:
"""
Generates right-hemisphere DKT cortical surface maps showing disease-specific partial Spearman ρ values from the cortical thickness–DGM association analysis.

For each diagnostic group, the script converts cortical region names to ggseg-compatible right-hemisphere ROI labels, maps correlation coefficients onto the DKT atlas, overlays FDR-corrected significance values, adds a labeled colorbar, and saves each figure as a 600-DPI PNG.
"""

# Disease groups to loop over
disease_groups = [
    "alzheimer's disease",
    "lewy body disease",
    "ftld-tdp",
    "tauopathies"
]

for disease in disease_groups:
    print(f"\nPlotting DKT map for: {disease}")

    # ------------------------------------------------------------
    # 1. Subset results for this disease
    # ------------------------------------------------------------
    df_g = results_df[results_df["Disease"] == disease].copy()
    if df_g.empty:
        print(f"⚠️ No data for {disease}, skipping.")
        continue

    # ------------------------------------------------------------
    # 2. Convert cortical ROI → DKT 'region_right' names
    # ------------------------------------------------------------
    df_g["roi_right"] = df_g["Cortical"] + "_right"
    df_g["value"]     = df_g["rho"]
    df_g["pval"]      = df_g["p_FDR"]

    # Dicts for ggseg
    data_map   = dict(zip(df_g["roi_right"], df_g["value"]))
    p_vals_map = dict(zip(df_g["roi_right"], df_g["pval"]))

    # ------------------------------------------------------------
    # 3. Draw the DKT map
    # ------------------------------------------------------------
    fig = plt.figure(figsize=(10, 10))

    plot_dk(
        data_map,              # ρ values
        p_vals_map,            # p-values (FDR)
        cmap='PRGn',
        vminmax=[-1, 1],
        background='white',
        fontsize=22,
        title=f"Right-Hemisphere DKT — ρ values\n{disease}"
    )

    # ------------------------------------------------------------
    # 4. FORCE A COLORBAR WITH TICK LABELS
    # ------------------------------------------------------------
    # find the mappable (the surface object ggseg uses)
    mappable = None
    for obj in fig.get_children():
        if hasattr(obj, "get_array"):
            mappable = obj
            break

    if mappable is not None:
        cbar = fig.colorbar(
            mappable,
            ax=plt.gca(),
            fraction=0.046,
            pad=0.04
        )
        cbar.set_label("ρ value", fontsize=18)
        cbar.ax.tick_params(labelsize=14)

    # ------------------------------------------------------------
    # 5. SAVE AS 600-DPI PNG
    # ------------------------------------------------------------
    outname = f"dkt_{disease.replace(' ', '_')}_subcortical_only_no_nucleus.png"
    plt.savefig(outname, dpi=600, bbox_inches="tight")
    print(f"Saved: {outname}")

    plt.show()


In [ ]:
"""
Generates DKT regional cortical thickness boxplots comparing postmortem cortical thickness across four neuropathological disease groups.

The script performs covariate-adjusted pairwise likelihood-ratio tests for each cortical region, applies FDR correction across comparisons, annotates significant group differences on 7×5 regional box/strip plots, and saves the final figure as both PDF and PNG.
"""

df_use = df_use.copy()
df_use["NPDx1"] = df_use["NPDx1"].astype(str).str.strip().str.lower()

order = ["alzheimer's disease", "lewy body disease", "ftld-tdp", "tauopathies"]

x_labels = [
    "AD",
    "LBD",
    "FTLD-TDP",
    "FTLD-Tau"
]

covars = ["AgeatDeath", "Sex", "Education", "PMI"]
palette = ["#3366CC", "#DC3912", "#109618", "#FF9900"]

sns.set(style="whitegrid", context="talk", font_scale=1.0)

# ============================================================
# DKT cortical regions
# ============================================================

cortical_regions = [
    'bankssts','caudalanteriorcingulate','caudalmiddlefrontal',
    'corpuscallosum','cuneus','entorhinal','fusiform','inferiorparietal',
    'inferiortemporal','isthmuscingulate','lateraloccipital',
    'lateralorbitofrontal','lingual','medialorbitofrontal',
    'middletemporal','parahippocampal','paracentral','parsopercularis',
    'parsorbitalis','parstriangularis','pericalcarine','postcentral',
    'posteriorcingulate','precentral','precuneus','rostralanteriorcingulate',
    'rostralmiddlefrontal','superiorfrontal','superiorparietal',
    'superiortemporal','supramarginal','frontalpole','temporalpole',
    'transversetemporal','insula'
]

# Keep only existing columns
cortical_regions = [c for c in cortical_regions if c in df_use.columns]

print("Regions found:", len(cortical_regions))
print(cortical_regions)

# ============================================================
# Keep needed columns
# ============================================================

needed = cortical_regions + ["NPDx1"] + [c for c in covars if c in df_use.columns]
df_plot = df_use[needed].dropna().copy()

# ============================================================
# Pairwise covariate-adjusted LRTs
# ============================================================

pairwise_results = []

for r in cortical_regions:
    ycol = r

    for g1, g2 in combinations(order, 2):
        d = df_plot[df_plot["NPDx1"].isin([g1, g2])].copy()
        if len(d) < 10:
            continue

        covars_here = [c for c in covars if c in d.columns]

        reduced = ols(f"{ycol} ~ " + " + ".join(covars_here), data=d).fit()
        full = ols(f"{ycol} ~ C(NPDx1) + " + " + ".join(covars_here), data=d).fit()

        lr = 2 * (full.llf - reduced.llf)
        df_diff = full.df_model - reduced.df_model
        p = chi2.sf(lr, df_diff)

        pairwise_results.append({
            "Region": r,
            "Group1": g1,
            "Group2": g2,
            "p_raw": p
        })

pairwise_df = pd.DataFrame(pairwise_results)

if not pairwise_df.empty:
    reject, p_corr = pg.multicomp(pairwise_df["p_raw"], method="fdr_bh")
    pairwise_df["p_FDR"] = p_corr
    pairwise_df["Sig"] = pairwise_df["p_FDR"].apply(
        lambda p: "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
    )

# ============================================================
# Plotting
# ============================================================

ncols = 5
nrows = 7

fig, axes = plt.subplots(nrows, ncols, figsize=(4.8*ncols, 4.2*nrows), sharey=False)
axes = np.array(axes).reshape(-1)

def plot_region(ax, r):
    ycol = r
    d = df_plot[["NPDx1", ycol]].dropna().copy()

    if d.empty:
        ax.axis("off")
        return

    sns.boxplot(
        data=d, x="NPDx1", y=ycol,
        order=order,
        palette=palette,
        ax=ax,
        width=0.55,
        fliersize=0,
        linewidth=1.2
    )

    sns.stripplot(
        data=d, x="NPDx1", y=ycol,
        order=order,
        color="black",
        size=3,
        alpha=0.5,
        ax=ax,
        jitter=0.15
    )

    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticklabels(x_labels, fontsize=12)

    ymin, ymax = d[ycol].min(), d[ycol].max()
    yr = ymax - ymin if ymax > ymin else 1.0
    ax.set_ylim(ymin - 0.08*yr, ymax + 0.35*yr)

    ax.set_title(r, fontsize=14, fontweight="bold")
    ax.grid(axis="y", linestyle=":", alpha=0.4)

    if not pairwise_df.empty:
        pairs = pairwise_df[pairwise_df["Region"] == r]
        y_offset = 0.025 * yr
        y_pos = ymax + 0.08 * yr

        for _, row in pairs.iterrows():
            if row["Sig"]:
                x1 = order.index(row["Group1"])
                x2 = order.index(row["Group2"])

                ax.plot(
                    [x1, x1, x2, x2],
                    [y_pos, y_pos+y_offset, y_pos+y_offset, y_pos],
                    lw=1.1, color="black"
                )

                ax.text(
                    (x1+x2)/2, y_pos + y_offset*0.8,
                    row["Sig"],
                    ha="center", fontsize=8, fontweight="bold"
                )

                y_pos += y_offset * 1.8

for ax, r in zip(axes, cortical_regions):
    plot_region(ax, r)

for ax in axes[len(cortical_regions):]:
    ax.axis("off")

fig.text(0.01, 0.5, "Mean cortical thickness (mm)", va="center", rotation="vertical",
         fontsize=20, fontweight="bold")
fig.text(0.5, 0.02, "Disease groups", ha="center",
         fontsize=20, fontweight="bold")

fig.suptitle(
    "Postmortem regional cortical thickness differentiates neuropathological groups",
    fontsize=20, fontweight="bold"
)

plt.tight_layout(rect=[0.03, 0.04, 1, 0.97])
plt.savefig("postmortem_cortical_DKT_thickness_group_boxplots.pdf", dpi=400, bbox_inches="tight")
plt.savefig("postmortem_cortical_DKT_thickness_group_boxplots.png", dpi=400, bbox_inches="tight")

plt.show()

In [ ]:
##########
##########
#####   AD/LBD  #####
##########
##########

df_use["NPDx1"] = df_use["NPDx1"].astype(str).str.strip().str.lower()
df_use["NPDx2"] = df_use["NPDx2"].astype(str).str.strip().str.lower()

df_ad_lbd = df_use[
    (df_use["NPDx1"] == "alzheimer's disease") &
    (df_use["NPDx2"] == "lewy body disease")
].copy()

df_lbd_ad = df_use[
    (df_use["NPDx1"] == "lewy body disease") &
    (df_use["NPDx2"] == "alzheimer's disease")
].copy()

print("\nAD + LBD")
print(df_ad_lbd[["INDDID", "NPDx1", "NPDx2"]])
print("N =", len(df_ad_lbd))

print("\nLBD + AD")
print(df_lbd_ad[["INDDID", "NPDx1", "NPDx2"]])
print("N =", len(df_lbd_ad))



In [ ]:
"""
Compares ICV-normalized postmortem subcortical volumes between reviewer-defined mixed-pathology groups: AD-primary/LBD-secondary versus LBD-primary/AD-secondary.

The script normalizes selected limbic and subcortical volumes by antemortem ICV, performs covariate-adjusted likelihood-ratio tests with FDR correction, and generates two-row box/strip plots with significance annotations.
"""

# ============================================================
# 1. Build 2-group dataframe
# ============================================================

df_ad_lbd = df_ad_lbd.copy()
df_lbd_ad = df_lbd_ad.copy()

df_ad_lbd["ReviewerGroup"] = "AD (primary)\nLBD (secondary)"
df_lbd_ad["ReviewerGroup"] = "LBD (primary)\nAD (secondary)"

df_two = pd.concat([df_ad_lbd, df_lbd_ad], ignore_index=True)

order = [
    "AD (primary)\nLBD (secondary)",
    "LBD (primary)\nAD (secondary)"
]

palette = ["#8E63CE", "#4CAF50"]  # purple, green
covars = ["AgeatDeath", "Sex", "Education", "PMI"]

sns.set(style="whitegrid", context="talk", font_scale=1.0)

# ============================================================
# 2. Subcortical structures
# ============================================================

subcortical_regions = [
    "hippocampus", "amygdala", "accumbens_area",
    "thalamus", "caudate", "putamen", "pallidum"
]

# keep only ones that exist
subcortical_regions = [
    s for s in subcortical_regions
    if f"postmortem_{s}" in df_two.columns
]

print("Subcortical regions found:", subcortical_regions)

# normalize by ICV
if "antemortem_icv" in df_two.columns:
    for s in subcortical_regions:
        col = f"postmortem_{s}"
        df_two[f"{s}_norm"] = df_two[col] / df_two["antemortem_icv"]
else:
    raise ValueError("Column 'antemortem_icv' not found in df_two.")

# ============================================================
# 3. Test each region
# ============================================================

results = []

for s in subcortical_regions:
    ycol = f"{s}_norm"
    needed = ["ReviewerGroup", ycol] + [c for c in covars if c in df_two.columns]
    d = df_two[needed].dropna().copy()

    if len(d) < 8 or d["ReviewerGroup"].nunique() < 2:
        continue

    covars_here = [c for c in covars if c in d.columns]

    if len(covars_here) == 0:
        reduced = ols(f"{ycol} ~ 1", data=d).fit()
        full = ols(f"{ycol} ~ C(ReviewerGroup)", data=d).fit()
    else:
        reduced = ols(f"{ycol} ~ " + " + ".join(covars_here), data=d).fit()
        full = ols(f"{ycol} ~ C(ReviewerGroup) + " + " + ".join(covars_here), data=d).fit()

    lr = 2 * (full.llf - reduced.llf)
    df_diff = full.df_model - reduced.df_model
    p = chi2.sf(lr, df_diff)

    results.append({
        "Region": s,
        "p_raw": p
    })

res_df = pd.DataFrame(results)

if not res_df.empty:
    from pingouin import multicomp
    _, p_corr = multicomp(res_df["p_raw"], method="fdr_bh")
    res_df["p_FDR"] = p_corr
    res_df["Sig"] = res_df["p_FDR"].apply(
        lambda p: "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
    )
else:
    res_df["p_FDR"] = []
    res_df["Sig"] = []

print(res_df)

# ============================================================
# 4. Plot: 3 panels on row 1, 4 panels on row 2
# ============================================================

fig = plt.figure(figsize=(20, 12))
gs = fig.add_gridspec(2, 12)

axes = [
    fig.add_subplot(gs[0, 0:4]),
    fig.add_subplot(gs[0, 4:8]),
    fig.add_subplot(gs[0, 8:12]),

    fig.add_subplot(gs[1, 0:3]),
    fig.add_subplot(gs[1, 3:6]),
    fig.add_subplot(gs[1, 6:9]),
    fig.add_subplot(gs[1, 9:12]),
]

def plot_region(ax, s):
    ycol = f"{s}_norm"
    needed = ["ReviewerGroup", ycol]
    d = df_two[needed].dropna().copy()

    if d.empty:
        ax.axis("off")
        return

    sns.boxplot(
        data=d, x="ReviewerGroup", y=ycol,
        order=order, palette=palette,
        width=0.55, fliersize=0, linewidth=1.2, ax=ax
    )

    sns.stripplot(
        data=d, x="ReviewerGroup", y=ycol,
        order=order, color="black",
        size=4, alpha=0.55, jitter=0.15, ax=ax
    )

    ax.tick_params(axis="x", labelsize=16)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title(s.replace("_", " ").capitalize(), fontsize=18, fontweight="bold")
    ax.grid(axis="y", linestyle=":", alpha=0.4)

    ymin, ymax = d[ycol].min(), d[ycol].max()
    yr = ymax - ymin if ymax > ymin else 1.0
    ax.set_ylim(ymin - 0.08 * yr, ymax + 0.22 * yr)

    row = res_df[res_df["Region"] == s]
    if not row.empty and row["Sig"].iloc[0] != "":
        sig = row["Sig"].iloc[0]
        y = ymax + 0.08 * yr
        h = 0.03 * yr
        ax.plot([0, 0, 1, 1], [y, y + h, y + h, y], lw=1.2, color="black")
        ax.text(
            0.5, y + h * 1.1, sig,
            ha="center", va="bottom",
            fontsize=16, fontweight="bold"
        )

# plot only available regions
for ax, s in zip(axes, subcortical_regions):
    plot_region(ax, s)

# turn off any unused axes if fewer than 7 regions are available
for ax in axes[len(subcortical_regions):]:
    ax.axis("off")

fig.text(
    0.01, 0.5, "ICV-normalized volume",
    va="center", rotation="vertical",
    fontsize=16, fontweight="bold"
)
fig.text(
    0.5, 0.02, "Reviewer-defined mixed pathology groups",
    ha="center", fontsize=16, fontweight="bold"
)

fig.suptitle(
    "Subcortical volumes: AD→LBD vs LBD→AD",
    fontsize=18, fontweight="bold"
)

plt.tight_layout(rect=[0.03, 0.04, 1, 0.96])
fig.suptitle(
    "Subcortical volumes: AD→LBD vs LBD→AD",
    fontsize=18, fontweight="bold"
)

plt.tight_layout(rect=[0.03, 0.04, 1, 0.96])

out_png = "subcortical_volumes_AD_primary_vs_LBD_primary.png"
plt.savefig(out_png, dpi=600, bbox_inches="tight")

plt.show()
print(f"Saved figure to: {out_png}")

In [ ]:
"""
Compares regional postmortem cortical thickness between reviewer-defined mixed-pathology groups: AD-primary/LBD-secondary versus LBD-primary/AD-secondary.

The script performs covariate-adjusted likelihood-ratio tests for each DKT cortical region, applies FDR correction across regions, and generates 7×5 cortical thickness box/strip plots with significance annotations saved as PNG and PDF.
"""

# ============================================================
# 1. Build 2-group dataframe
# ============================================================

df_ad_lbd = df_ad_lbd.copy()
df_lbd_ad = df_lbd_ad.copy()

df_ad_lbd["ReviewerGroup"] = "AD (primary)\nLBD (secondary)"
df_lbd_ad["ReviewerGroup"] = "LBD (primary)\nAD (secondary)"

df_two = pd.concat([df_ad_lbd, df_lbd_ad], ignore_index=True)

order = [
    "AD (primary)\nLBD (secondary)",
    "LBD (primary)\nAD (secondary)"
]

palette = ["#8E63CE", "#4CAF50"]  # purple, green
covars = ["AgeatDeath", "Sex", "Education", "PMI"]

sns.set(style="whitegrid", context="talk", font_scale=1.0)

# ============================================================
# 2. Cortical regions
# ============================================================

cortical_regions = [
    'bankssts','caudalanteriorcingulate','caudalmiddlefrontal',
    'corpuscallosum','cuneus','entorhinal','fusiform','inferiorparietal',
    'inferiortemporal','isthmuscingulate','lateraloccipital',
    'lateralorbitofrontal','lingual','medialorbitofrontal',
    'middletemporal','parahippocampal','paracentral','parsopercularis',
    'parsorbitalis','parstriangularis','pericalcarine','postcentral',
    'posteriorcingulate','precentral','precuneus','rostralanteriorcingulate',
    'rostralmiddlefrontal','superiorfrontal','superiorparietal',
    'superiortemporal','supramarginal','frontalpole','temporalpole',
    'transversetemporal','insula'
]

cortical_regions = [c for c in cortical_regions if c in df_two.columns]

print("Cortical regions found:", len(cortical_regions))
print(cortical_regions)

# ============================================================
# 3. Test each region
# ============================================================

results = []

for r in cortical_regions:
    needed = ["ReviewerGroup", r] + [c for c in covars if c in df_two.columns]
    d = df_two[needed].dropna().copy()

    if len(d) < 8 or d["ReviewerGroup"].nunique() < 2:
        continue

    covars_here = [c for c in covars if c in d.columns]

    if len(covars_here) == 0:
        reduced = ols(f"{r} ~ 1", data=d).fit()
        full = ols(f"{r} ~ C(ReviewerGroup)", data=d).fit()
    else:
        reduced = ols(f"{r} ~ " + " + ".join(covars_here), data=d).fit()
        full = ols(f"{r} ~ C(ReviewerGroup) + " + " + ".join(covars_here), data=d).fit()

    lr = 2 * (full.llf - reduced.llf)
    df_diff = full.df_model - reduced.df_model
    p = chi2.sf(lr, df_diff)

    results.append({
        "Region": r,
        "p_raw": p
    })

res_df = pd.DataFrame(results)

if not res_df.empty:
    _, p_corr = multicomp(res_df["p_raw"], method="fdr_bh")
    res_df["p_FDR"] = p_corr
    res_df["Sig"] = res_df["p_FDR"].apply(
        lambda p: "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
    )
else:
    res_df["p_FDR"] = []
    res_df["Sig"] = []

print(res_df)

# ============================================================
# 4. Plot
# ============================================================

ncols = 5
nrows = 7

fig, axes = plt.subplots(nrows, ncols, figsize=(4.8*ncols, 4.2*nrows), sharey=False)
axes = np.array(axes).reshape(-1)

def plot_region(ax, r):
    d = df_two[["ReviewerGroup", r]].dropna().copy()

    if d.empty:
        ax.axis("off")
        return

    sns.boxplot(
        data=d, x="ReviewerGroup", y=r,
        order=order, palette=palette,
        width=0.55, fliersize=0, linewidth=1.2, ax=ax
    )

    sns.stripplot(
        data=d, x="ReviewerGroup", y=r,
        order=order, color="black",
        size=3, alpha=0.55, jitter=0.15, ax=ax
    )

    ax.tick_params(axis="x", labelsize=14)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title(r, fontsize=16, fontweight="bold")
    ax.grid(axis="y", linestyle=":", alpha=0.4)

    ymin, ymax = d[r].min(), d[r].max()
    yr = ymax - ymin if ymax > ymin else 1.0
    ax.set_ylim(ymin - 0.08*yr, ymax + 0.22*yr)

    row = res_df[res_df["Region"] == r]
    if not row.empty and row["Sig"].iloc[0] != "":
        sig = row["Sig"].iloc[0]
        y = ymax + 0.08*yr
        h = 0.03*yr
        ax.plot([0, 0, 1, 1], [y, y+h, y+h, y], lw=1.2, color="black")
        ax.text(0.5, y+h*1.1, sig, ha="center", va="bottom", fontsize=18, fontweight="bold")

for ax, r in zip(axes, cortical_regions):
    plot_region(ax, r)

for ax in axes[len(cortical_regions):]:
    ax.axis("off")

fig.text(0.01, 0.5, "Regional mean cortical thickness (mm)", va="center", rotation="vertical",
         fontsize=16, fontweight="bold")
fig.text(0.5, 0.02, "AD vs LBD groups", ha="center",
         fontsize=16, fontweight="bold")

fig.suptitle("Postmortem regional cortical thickness differentiates neuropathological groups: AD vs LBD", fontsize=18, fontweight="bold")
plt.tight_layout(rect=[0.03, 0.04, 1, 0.96])

out_png = "cortical_thickness_AD_primary_vs_LBD_primary.png"
plt.savefig(out_png, dpi=600, bbox_inches="tight")

out_png = "cortical_thickness_AD_primary_vs_LBD_primary.pdf"
plt.savefig(out_png, dpi=600, bbox_inches="tight")

plt.show()
print(f"Saved figure to: {out_png}")

In [ ]:
"""
Generates a right-hemisphere DKT cortical map of AD-primary/LBD-secondary versus LBD-primary/AD-secondary group differences using -log10(raw p-values).

The script performs covariate-adjusted likelihood-ratio tests for each cortical region, applies FDR correction for significance masking, exports the raw/FDR p-value table, and visualizes regional statistical strength on a green DKT surface map with a horizontal colorbar.
"""

# ============================================================
# 1. Build 2-group dataframe
# ============================================================

df_ad_lbd = df_ad_lbd.copy()
df_lbd_ad = df_lbd_ad.copy()

df_ad_lbd["ReviewerGroup"] = "AD (primary)\nLBD (secondary)"
df_lbd_ad["ReviewerGroup"] = "LBD (primary)\nAD (secondary)"

df_two = pd.concat([df_ad_lbd, df_lbd_ad], ignore_index=True)

order = [
    "AD (primary)\nLBD (secondary)",
    "LBD (primary)\nAD (secondary)"
]

palette = ["#8E63CE", "#4CAF50"]
covars = ["AgeatDeath", "Sex", "Education", "PMI"]

sns.set(style="whitegrid", context="talk", font_scale=1.0)

# ============================================================
# 2. Cortical regions
# ============================================================

cortical_regions = [
    'bankssts','caudalanteriorcingulate','caudalmiddlefrontal',
    'corpuscallosum','cuneus','entorhinal','fusiform','inferiorparietal',
    'inferiortemporal','isthmuscingulate','lateraloccipital',
    'lateralorbitofrontal','lingual','medialorbitofrontal',
    'middletemporal','parahippocampal','paracentral','parsopercularis',
    'parsorbitalis','parstriangularis','pericalcarine','postcentral',
    'posteriorcingulate','precentral','precuneus','rostralanteriorcingulate',
    'rostralmiddlefrontal','superiorfrontal','superiorparietal',
    'superiortemporal','supramarginal','frontalpole','temporalpole',
    'transversetemporal','insula'
]

cortical_regions = [c for c in cortical_regions if c in df_two.columns]

print("Cortical regions found:", len(cortical_regions))
print(cortical_regions)

# ============================================================
# 3. Likelihood-ratio test for each region
# ============================================================

results = []

for r in cortical_regions:
    needed = ["ReviewerGroup", r] + [c for c in covars if c in df_two.columns]
    d = df_two[needed].dropna().copy()

    if len(d) < 8 or d["ReviewerGroup"].nunique() < 2:
        continue

    covars_here = [c for c in covars if c in d.columns]

    if len(covars_here) == 0:
        reduced = ols(f"{r} ~ 1", data=d).fit()
        full    = ols(f"{r} ~ C(ReviewerGroup)", data=d).fit()
    else:
        reduced = ols(f"{r} ~ " + " + ".join(covars_here), data=d).fit()
        full    = ols(f"{r} ~ C(ReviewerGroup) + " + " + ".join(covars_here), data=d).fit()

    lr = 2 * (full.llf - reduced.llf)
    df_diff = full.df_model - reduced.df_model
    p = chi2.sf(lr, df_diff)

    results.append({
        "Region": r,
        "p_raw": p,
        "N": len(d)
    })

res_df = pd.DataFrame(results)

if not res_df.empty:
    _, p_corr = multicomp(res_df["p_raw"], method="fdr_bh")
    res_df["p_FDR"] = p_corr
    res_df["Sig"] = res_df["p_FDR"].apply(
        lambda p: "***" if p < 0.001 else
                  "**"  if p < 0.01 else
                  "*"   if p < 0.05 else ""
    )
else:
    res_df["p_FDR"] = []
    res_df["Sig"] = []

# ============================================================
# 4. Compute -log10(raw p)
# ============================================================

eps = np.finfo(float).tiny
res_df["p_raw_safe"] = res_df["p_raw"].clip(lower=eps)
res_df["neglog10_p"] = -np.log10(res_df["p_raw_safe"])

print(res_df[["Region", "p_raw", "p_FDR", "Sig", "neglog10_p"]])

# ============================================================
# 5. Prepare data for ggseg / plot_dk
# ============================================================

# If corpuscallosum causes trouble in DKT plotting, uncomment next line:
# res_df = res_df[res_df["Region"] != "corpuscallosum"].copy()

res_df["roi_right"] = res_df["Region"] + "_right"
res_df["value"] = res_df["neglog10_p"]
res_df["pval"] = res_df["p_FDR"]

data_map   = dict(zip(res_df["roi_right"], res_df["value"]))
p_vals_map = dict(zip(res_df["roi_right"], res_df["pval"]))

# ============================================================
# 6. Plot DKT map
# ============================================================

outname = "RightHemisphere_DKT_neglog10_rawP_AD_vs_LBD_Greens.png"

vmax = float(np.ceil(res_df["neglog10_p"].max()))
vmax = max(vmax, 3.0)
vmin = 0.0

fig = plt.figure(figsize=(10, 10))

plot_dk(
    data_map,
    p_vals_map,
    cmap="Greens",
    vminmax=[vmin, vmax],
    background="white",
    fontsize=22,
    title="Right-Hemisphere DKT\n-log10(raw p), significance from FDR"
)

# ============================================================
# 7. Force a numeric HORIZONTAL colorbar legend
# ============================================================

norm = Normalize(vmin=vmin, vmax=vmax)
sm = ScalarMappable(norm=norm, cmap=plt.get_cmap("Greens"))
sm.set_array([])

ticks = np.arange(vmin, vmax + 1, 1)

cbar = fig.colorbar(
    sm,
    ax=plt.gca(),
    orientation="horizontal",
    fraction=0.06,
    pad=0.08,
    ticks=ticks
)

cbar.set_label("-log10(raw p-value)", fontsize=44)
cbar.ax.tick_params(labelsize=36)
cbar.ax.set_xticklabels([f"{int(t)}" for t in ticks])

# ============================================================
# 8. Save
# ============================================================

plt.savefig(outname, dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
############################
############################
"""
For the 5 groups, the groups to be useed are:
order = [
    "alzheimer's disease",
    "lewy body disease",
    "ftld-tdp",
    "3R-tau",
    "4R-tau"
]
"""